In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
import pandas as pd

In [2]:
class MyLogisticRegression(BaseEstimator, ClassifierMixin):
    def __init__ (self, learning_rate: float = 0.01, epoch: int = 1000, lambda_: float = 0.00, penalty: str = None):
        self.learning_rate = learning_rate
        self.epoch = epoch
        self.lambda_ = lambda_
        self.penalty = penalty

    def mySigmoid(self, x: np.ndarray) -> np.ndarray:
        return 1/(1 + np.exp(-x))

    def myBinaryCrossEntropy(self, y_true: np.ndarray, y_pred: np.ndarray, w: np.ndarray) -> float:
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        base_loss = -np.mean(y_true* np.log(y_pred) + (1 - y_true)* np.log(1 - y_pred))
        n = len(y_true)
        reg = self.penalty.lower() if isinstance(self.penalty, str) else None
        match reg:
            case 'l1':
                reg_loss = (self.lambda_ / n) * np.sum(np.abs(w))
                return base_loss + reg_loss
            case 'l2':
                reg_loss = (self.lambda_ / (2 * n)) * np.sum(w ** 2)
                return base_loss + reg_loss
            case _:
                return base_loss
            
        
    def MylinearRegression(self, X: np.ndarray, w: np.ndarray, b: float ) -> np.ndarray:
        return np.dot(X,w) + b

    def predict(self, X: np.ndarray, threshold: float = 0.5 ) -> np.ndarray:
        linear_output = self.MylinearRegression(X, self.w_, self.b_)
        y_probability = self.mySigmoid(linear_output)
        return np.array([1 if p >= threshold else 0 for p in y_probability])

    def fit(self, X: np.ndarray, y: np.ndarray):
        reg = self.penalty.lower() if isinstance(self.penalty, str) else None
        n, n_features = X.shape
        self.w_ = np.zeros(n_features)
        self.b_ = 0.00
        self.loss_ = []

        for ep in range(self.epoch):
            linear_output = self.MylinearRegression(X, self.w_, self.b_)
            y_pred = self.mySigmoid(linear_output)

            current_loss = self.myBinaryCrossEntropy(y, y_pred, self.w_)
            self.loss_.append(current_loss)

            dw = (1 / n) * np.dot(X.T, (y_pred - y))
            db = (1 / n) * np.sum(y_pred - y)

            if reg == 'l1':
                dw += (self.lambda_ / n) * np.sign(self.w_)
            elif reg == 'l2':
                dw += (self.lambda_ / n) * self.w_

            self.w_ -= self.learning_rate * dw
            self.b_ -= self.learning_rate * db

            if (ep + 1) % 50 == 0:
                 w_str = "[" + ", ".join([f"{w:.2f}" for w in self.w_]) + "]"
                 print(f'Epoch: {ep+1:4} || weights: {w_str} || bias: {self.b_:.2f} || loss: {self.loss_[ep]:.2f}')

        return self

In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=1000, n_features=2, n_classes=2, n_redundant=0, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = MyLogisticRegression(learning_rate=0.001, epoch = 50000)
model.fit(X_train, y_train)

Epoch:   50 || weights: [-0.00, 0.02] || bias: 0.00 || loss: 0.68
Epoch:  100 || weights: [-0.00, 0.05] || bias: 0.00 || loss: 0.67
Epoch:  150 || weights: [-0.00, 0.07] || bias: 0.00 || loss: 0.66
Epoch:  200 || weights: [-0.00, 0.09] || bias: 0.00 || loss: 0.65
Epoch:  250 || weights: [-0.00, 0.12] || bias: 0.00 || loss: 0.64
Epoch:  300 || weights: [-0.01, 0.14] || bias: 0.00 || loss: 0.63
Epoch:  350 || weights: [-0.01, 0.16] || bias: 0.00 || loss: 0.62
Epoch:  400 || weights: [-0.01, 0.18] || bias: 0.00 || loss: 0.61
Epoch:  450 || weights: [-0.01, 0.20] || bias: 0.00 || loss: 0.60
Epoch:  500 || weights: [-0.01, 0.22] || bias: 0.00 || loss: 0.59
Epoch:  550 || weights: [-0.01, 0.24] || bias: 0.00 || loss: 0.59
Epoch:  600 || weights: [-0.01, 0.26] || bias: 0.00 || loss: 0.58
Epoch:  650 || weights: [-0.01, 0.28] || bias: 0.00 || loss: 0.57
Epoch:  700 || weights: [-0.01, 0.30] || bias: 0.00 || loss: 0.57
Epoch:  750 || weights: [-0.02, 0.31] || bias: 0.00 || loss: 0.56
Epoch:  80

,learning_rate,0.001
,epoch,50000
,lambda_,0.0
,penalty,None
Name,Type,Value
b_,float64,0.2179
lambda_,float,0
loss_,list,"[np.float64(0.6931471805599452), np.float64(0.6928983038581674), np.float64(0.6926496734436164), np.float64(0.6924012890698894), ...]"
w_,"ndarray[float64](2,)","[-0.32, 2.04]"


In [4]:
from sklearn.linear_model import LogisticRegression

In [5]:
sk_model = LogisticRegression(penalty=None) 
sk_model.fit(X_train, y_train)

c:\Users\offsh\Documents\project\tf_env\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",None
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver beca

In [6]:
predictions = model.predict(X_test)
my_acc = accuracy_score(y_test, predictions)
sk_predictions = sk_model.predict(X_test)
sk_acc = accuracy_score(y_test, sk_predictions)

In [7]:
print("\n--- PERFORMANCE COMPARISON ---")
print(f"Custom Model Accuracy: {my_acc * 100:.1f}%")
print(f"SKLearn Model Accuracy: {sk_acc * 100:.1f}%")

print("\n--- WEIGHTS (Slopes) ---")
print(f"Custom Weights:  {np.round(model.w_, 4)}")
print(f"SKLearn Weights: {np.round(sk_model.coef_[0], 4)}")

print("\n--- BIAS (Intercept) ---")
print(f"Custom Bias:  {model.b_:.4f}")
print(f"SKLearn Bias: {sk_model.intercept_[0]:.4f}")


--- PERFORMANCE COMPARISON ---
Custom Model Accuracy: 88.0%
SKLearn Model Accuracy: 88.0%

--- WEIGHTS (Slopes) ---
Custom Weights:  [-0.3246  2.04  ]
SKLearn Weights: [-0.3354  2.0786]

--- BIAS (Intercept) ---
Custom Bias:  0.2179
SKLearn Bias: 0.2299


In [8]:
import joblib

In [9]:
joblib.dump(model, 'my_custom_logisticRegressionModel.pkl')
joblib.dump(sk_model, 'sklearn_logisticRegressionModel.pkl')
print("Model saved successfully!")

Model saved successfully!
